## Summary

### What we produced:
1. **3 trained neural networks** — one per aircraft type (X-47B, Cessna 172, MQ-9 Reaper)
2. **PyTorch model files** — `.pt` checkpoints for local FastAPI serving
3. **Feature scalers** — `.pkl` files for consistent preprocessing at inference
4. **MLflow Model Registry** — models registered for Domino deployment

### Deployment Options:

**Option A: Domino Model Registry (Production)**
- Models registered via `mlflow.pytorch.log_model()` with `registered_model_name`
- Deploy via Domino UI: Publish → Model APIs → select registered model
- Domino serves via the `model_api.py` predict function (see `packages/ml/model_api.py`)
- Endpoint: `https://<domino>/models/<model_id>/<version>/predict`

**Option B: Local FastAPI (Development)**
- Run `packages/ml/model_server.py` — loads PyTorch models directly
- Serves on port 5051: `POST /wind-correct`
- No Domino dependency, works anywhere

### Next steps:
1. For Domino: transition models to Production stage in MLflow UI
2. For local: run `python model_server.py` and connect from the flight sim
3. Enable wind physics + ML correction toggle in the flight simulator app

In [ ]:
# ============================================================
# PLOT: Training Loss Curves — All Aircraft
# ============================================================
fig, axes = plt.subplots(1, len(AIRCRAFT_IDS), figsize=(18, 5))
fig.patch.set_facecolor('#0a1628')
fig.suptitle('TRAINING CONVERGENCE — PER AIRCRAFT MODEL', 
             color='#00e5ff', fontsize=14, fontfamily='monospace', y=1.02)

colors = {'x47b': '#00e5ff', 'cessna172': '#ffc107', 'mq9_reaper': '#00e676'}

for idx, aircraft_id in enumerate(AIRCRAFT_IDS):
    ax = axes[idx]
    ax.set_facecolor('#0d1f3c')
    
    results = all_training_results[aircraft_id]
    profile = get_profile(aircraft_id)
    
    ax.plot(results["train_losses"], color=colors[aircraft_id], alpha=0.7, label='Train')
    ax.plot(results["val_losses"], color='white', alpha=0.9, label='Validation')
    
    ax.set_title(f"{profile['name']}\nTest MSE: {results['test_loss']:.6f}", 
                 color=colors[aircraft_id], fontsize=10, fontfamily='monospace')
    ax.set_xlabel('Epoch', color='#8ba4c4', fontsize=9)
    ax.set_ylabel('MSE Loss', color='#8ba4c4', fontsize=9)
    ax.legend(facecolor='#0d1f3c', edgecolor='#1a3a5c', labelcolor='#8ba4c4', fontsize=8)
    ax.tick_params(colors='#4a6a8a', labelsize=8)
    ax.grid(True, alpha=0.1, color='#1a3a5c')
    ax.spines[:].set_color('#1a3a5c')

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/training_curves.png', dpi=150, facecolor='#0a1628', bbox_inches='tight')
plt.show()

# ============================================================
# PLOT: MAE Comparison Bar Chart
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0a1628')
ax.set_facecolor('#0d1f3c')

x = np.arange(3)
width = 0.25

for i, aircraft_id in enumerate(AIRCRAFT_IDS):
    mae = all_training_results[aircraft_id]["mae"]
    profile = get_profile(aircraft_id)
    bars = ax.bar(x + i * width, [mae["heading"], mae["throttle"], mae["pitch"]], 
                  width, label=profile["name"], color=colors[aircraft_id], alpha=0.8)

ax.set_xlabel('Correction Axis', color='#8ba4c4', fontsize=11)
ax.set_ylabel('Mean Absolute Error', color='#8ba4c4', fontsize=11)
ax.set_title('MODEL ACCURACY — PER AIRCRAFT COMPARISON', 
             color='#00e5ff', fontsize=13, fontfamily='monospace')
ax.set_xticks(x + width)
ax.set_xticklabels(['Heading (°)', 'Throttle (km/h)', 'Pitch (°)'])
ax.legend(facecolor='#0d1f3c', edgecolor='#1a3a5c', labelcolor='#8ba4c4')
ax.tick_params(colors='#4a6a8a')
ax.grid(True, alpha=0.1, color='#1a3a5c', axis='y')
ax.spines[:].set_color('#1a3a5c')

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/mae_comparison.png', dpi=150, facecolor='#0a1628')
plt.show()

print("\n→ Key demo point: Each aircraft has DIFFERENT error profiles.")
print("  The Cessna needs larger heading corrections (more wind-affected).")
print("  The X-47B needs precise throttle corrections (heavy, momentum-driven).")
print("  This is WHY we need per-aircraft models trained on a Ray cluster.")

## Part 4: Cross-Aircraft Model Comparison

This is the key demo visualization — showing that **each aircraft learned different corrections** from the same wind conditions. This proves why the Ray cluster is essential: you need separate training runs per aircraft type.

In [ ]:
# ============================================================
# MLFLOW SETUP
# ============================================================
# For Domino Data Lab, set the tracking URI:
# mlflow.set_tracking_uri("https://your-domino-instance.com/mlflow")

# For local development:
mlflow.set_tracking_uri("mlruns")

print("=" * 60)
print("TRAINING WIND CORRECTION MODELS")
print("=" * 60)

all_training_results = {}

for aircraft_id in AIRCRAFT_IDS:
    profile = get_profile(aircraft_id)
    
    print(f"\n{'─'*60}")
    print(f"AIRCRAFT: {profile['name']}")
    print(f"  Mass: {profile['mass']:,} kg | Drag: {profile['drag_coeff']} | Sensitivity: {profile['wind_sensitivity']}")
    print(f"{'─'*60}")
    
    # Set MLflow experiment for this aircraft
    experiment_name = f"drone-wind-correction-{aircraft_id}"
    mlflow.set_experiment(experiment_name)
    
    # Load and prepare data
    data = prepare_data(aircraft_id)
    
    # Start MLflow run
    with mlflow.start_run(run_name=f"{aircraft_id}_v1"):
        # Log aircraft metadata as tags
        mlflow.set_tags({
            "aircraft_id": aircraft_id,
            "aircraft_name": profile["name"],
            "wingspan": str(profile["wingspan"]),
            "mass": str(profile["mass"]),
            "drag_coeff": str(profile["drag_coeff"]),
            "wind_sensitivity": str(profile["wind_sensitivity"]),
        })
        
        # Log hyperparameters
        mlflow.log_params(HYPERPARAMS)
        
        # Train
        results = train_model(aircraft_id, data, HYPERPARAMS)
        all_training_results[aircraft_id] = results
        
        # Save model artifacts
        model = results["model"]
        scaler = results["scaler"]
        
        # 1. PyTorch model (.pt) — used by both local FastAPI and Domino Model API
        pt_path = f"{MODEL_DIR}/wind_correction_{aircraft_id}.pt"
        torch.save(model.state_dict(), pt_path)
        mlflow.log_artifact(pt_path)
        
        # 2. Feature scaler (.pkl) — required for preprocessing at inference time
        scaler_path = f"{MODEL_DIR}/feature_scaler_{aircraft_id}.pkl"
        with open(scaler_path, "wb") as f:
            pickle.dump(scaler, f)
        mlflow.log_artifact(scaler_path)
        
        # 3. Register full model in MLflow Model Registry
        #    This is what Domino Model Registry uses to deploy
        mlflow.pytorch.log_model(
            model, 
            "model",
            registered_model_name=f"wind-correction-{aircraft_id}"
        )
        
        print(f"  Artifacts saved: {pt_path}, {scaler_path}")
        print(f"  Registered in MLflow: wind-correction-{aircraft_id}")

print(f"\n{'='*60}")
print("ALL MODELS TRAINED AND REGISTERED")
print(f"{'='*60}")

## Part 3: Train All Aircraft Models

This cell trains one model per aircraft, tracked in MLflow. Each aircraft gets its own experiment for clean comparison in the MLflow UI.

**For Domino Data Lab**: Set `MLFLOW_TRACKING_URI` to your Domino MLflow endpoint.

In [ ]:
def train_model(aircraft_id: str, data: tuple, hyperparams: dict) -> dict:
    """
    Train a WindCorrectionNet for a specific aircraft type.
    
    Logs everything to MLflow:
    - Hyperparameters (lr, batch_size, epochs, hidden_dims)
    - Aircraft metadata (name, mass, wingspan, drag_coeff)
    - Per-epoch training and validation loss
    - Final test metrics (MSE, MAE per correction axis)
    - Model artifacts (PyTorch .pt, ONNX, scaler .pkl)
    
    Args:
        aircraft_id: Which aircraft to train for
        data: Tuple from prepare_data()
        hyperparams: Training hyperparameters dict
        
    Returns:
        Dict with training results and best model
    """
    X_train, y_train, X_val, y_val, X_test, y_test, scaler = data
    profile = get_profile(aircraft_id)
    
    # Convert to tensors
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_dataset = TensorDataset(
        torch.FloatTensor(X_train).to(device),
        torch.FloatTensor(y_train).to(device)
    )
    val_X = torch.FloatTensor(X_val).to(device)
    val_y = torch.FloatTensor(y_val).to(device)
    test_X = torch.FloatTensor(X_test).to(device)
    test_y = torch.FloatTensor(y_test).to(device)
    
    train_loader = DataLoader(train_dataset, batch_size=hyperparams["batch_size"], shuffle=True)
    
    # Initialize model
    model = WindCorrectionNet(
        hidden_dims=hyperparams["hidden_dims"],
        dropout=hyperparams["dropout"]
    ).to(device)
    
    optimizer = torch.optim.Adam(
        model.parameters(), 
        lr=hyperparams["learning_rate"],
        weight_decay=hyperparams["weight_decay"]
    )
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    # Training loop
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    best_model_state = None
    
    print(f"\n  Training {profile['name']} on {device}...")
    
    for epoch in range(hyperparams["epochs"]):
        # Train
        model.train()
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        train_loss = epoch_loss / len(train_loader)
        train_losses.append(train_loss)
        
        # Validate
        model.eval()
        with torch.no_grad():
            val_pred = model(val_X)
            val_loss = criterion(val_pred, val_y).item()
            val_losses.append(val_loss)
        
        scheduler.step(val_loss)
        
        # Track best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
        
        # Log to MLflow
        mlflow.log_metrics({
            "train_loss": train_loss,
            "val_loss": val_loss,
        }, step=epoch)
        
        if (epoch + 1) % 20 == 0:
            print(f"    Epoch {epoch+1:3d}/{hyperparams['epochs']}: "
                  f"train_loss={train_loss:.6f} val_loss={val_loss:.6f}")
    
    # Load best model and evaluate on test set
    model.load_state_dict(best_model_state)
    model.eval()
    
    with torch.no_grad():
        test_pred = model(test_X)
        test_loss = criterion(test_pred, test_y).item()
        
        # Per-axis MAE
        mae = torch.abs(test_pred - test_y).mean(dim=0)
        mae_heading = mae[0].item()
        mae_throttle = mae[1].item()
        mae_pitch = mae[2].item()
    
    # Log final metrics
    mlflow.log_metrics({
        "test_loss": test_loss,
        "best_val_loss": best_val_loss,
        "mae_heading": mae_heading,
        "mae_throttle": mae_throttle,
        "mae_pitch": mae_pitch,
    })
    
    print(f"\n  ✓ {profile['name']} Results:")
    print(f"    Test MSE: {test_loss:.6f}")
    print(f"    MAE — Heading: {mae_heading:.3f}° | Throttle: {mae_throttle:.3f} km/h | Pitch: {mae_pitch:.3f}°")
    
    return {
        "model": model,
        "scaler": scaler,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "test_loss": test_loss,
        "mae": {"heading": mae_heading, "throttle": mae_throttle, "pitch": mae_pitch},
    }


print("Training function defined.")

## Part 2: Training Loop with MLflow Tracking

Each aircraft gets its own MLflow experiment. This allows comparing:
- How different aircraft need different correction magnitudes
- Whether certain architectures work better for heavy vs light aircraft
- Training convergence rates across aircraft types

**MLflow tracks**: hyperparameters, loss curves, model artifacts, and registers the best model per aircraft.

In [ ]:
class WindCorrectionNet(nn.Module):
    """
    Neural network for predicting optimal flight corrections under wind conditions.
    
    Architecture: 
        Input(9) → Dense(128) → ReLU → Dropout → Dense(64) → ReLU → Dropout → 
        Dense(32) → ReLU → Dense(3) → Tanh → Scale
    
    The Tanh output is scaled to physical correction ranges:
        - heading_correction: [-15°, +15°]
        - throttle_correction: [-50, +50] km/h  
        - pitch_correction: [-10°, +10°]
    
    One model is trained PER AIRCRAFT TYPE because each aircraft has unique
    aerodynamic properties that change how it responds to wind.
    """
    
    def __init__(self, input_dim: int = 9, hidden_dims: list = [128, 64, 32], 
                 output_dim: int = 3, dropout: float = 0.1):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h_dim
        
        # Output layer with Tanh (bounds to [-1, 1])
        layers.extend([
            nn.Linear(prev_dim, output_dim),
            nn.Tanh(),
        ])
        
        self.network = nn.Sequential(*layers)
        
        # Scaling factors to convert Tanh [-1,1] to physical ranges
        self.register_buffer('output_scale', torch.tensor([
            CORRECTION_RANGES["heading"],
            CORRECTION_RANGES["throttle"],
            CORRECTION_RANGES["pitch"],
        ]))
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Tensor of shape (batch, 9) — normalized aircraft state + wind
            
        Returns:
            Tensor of shape (batch, 3) — [heading_corr, throttle_corr, pitch_corr]
        """
        raw = self.network(x)
        return raw * self.output_scale  # Scale from [-1,1] to physical ranges


def prepare_data(aircraft_id: str) -> tuple:
    """
    Load simulation data and prepare train/val/test splits.
    
    Key preprocessing:
    - Encode wind_dir as sin/cos (avoids discontinuity at 0°/360°)
    - Normalize features with StandardScaler
    - 70/15/15 split
    """
    path = f"{DATA_DIR}/simulation_results_{aircraft_id}.parquet"
    df = pd.read_parquet(path)
    
    # Feature engineering: encode wind direction as sin/cos
    df["wind_dir_sin"] = np.sin(np.radians(df["wind_dir"]))
    df["wind_dir_cos"] = np.cos(np.radians(df["wind_dir"]))
    
    X = df[FEATURE_COLS].values
    y = df[TARGET_COLS].values
    
    # Train/val/test split
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
    
    # Normalize features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    
    print(f"  {aircraft_id}: train={len(X_train):,} val={len(X_val):,} test={len(X_test):,}")
    
    return (X_train, y_train, X_val, y_val, X_test, y_test, scaler)

print("Model and data preparation functions defined.")

## Part 1: Model Definition

The `WindCorrectionNet` takes 9 input features (aircraft state + wind conditions) and outputs 3 correction values. The Tanh activation on the output layer naturally bounds corrections to a safe range.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import os
import pickle
import time

from aircraft_profiles import AIRCRAFT_PROFILES, get_profile, list_aircraft

# ============================================================
# CONFIGURATION
# ============================================================
AIRCRAFT_IDS = ["x47b", "cessna172", "mq9_reaper"]
DATA_DIR = "data"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Training hyperparameters
HYPERPARAMS = {
    "learning_rate": 1e-3,
    "batch_size": 256,
    "epochs": 100,
    "hidden_dims": [128, 64, 32],
    "dropout": 0.1,
    "weight_decay": 1e-5,
}

# Feature columns (input to the model)
FEATURE_COLS = [
    "actual_speed", "actual_heading", "actual_pitch", "actual_roll", 
    "actual_vertical_velocity",
    "wind_dir_sin", "wind_dir_cos",  # Encode direction as sin/cos (no discontinuity at 360°)
    "wind_speed", "turbulence"
]

# Target columns (what the model predicts)
TARGET_COLS = [
    "optimal_heading_correction",
    "optimal_throttle_correction", 
    "optimal_pitch_correction"
]

# Correction output ranges (for Tanh scaling)
CORRECTION_RANGES = {
    "heading": 15.0,     # degrees
    "throttle": 50.0,    # km/h
    "pitch": 10.0,       # degrees
}

print("Configuration loaded.")
print(f"  Features: {len(FEATURE_COLS)}")
print(f"  Targets: {len(TARGET_COLS)}")
print(f"  Epochs: {HYPERPARAMS['epochs']}")
print(f"  Batch size: {HYPERPARAMS['batch_size']}")

# 🧠 Wind Correction ML Model Training
## Per-Aircraft Neural Networks with MLflow Experiment Tracking

### Overview
This notebook trains a **separate neural network for each aircraft type** to predict optimal flight corrections given current state + wind conditions. Tracked via **MLflow** in Domino Data Lab.

### Why per-aircraft models?
A 20,000 kg X-47B stealth drone responds completely differently to a 30 m/s crosswind than a 1,100 kg Cessna 172. The correction model must learn aircraft-specific dynamics:
- **X-47B**: Small corrections, high inertia, stable but slow to respond
- **Cessna 172**: Large corrections, low inertia, agile but easily blown off course
- **MQ-9 Reaper**: Medium corrections, moderate inertia, long wings catch crosswinds

### Model Architecture
```
Input (9 features)           Hidden Layers              Output (3 corrections)
──────────────                ─────────────              ──────────────────────
speed                   ┌─→ Dense(128, ReLU) ─┐         heading_correction
heading                 │                      │         throttle_correction
pitch              ─────┤→ Dense(64, ReLU)  ──┤────→    pitch_correction
roll                    │                      │
vertical_velocity       └─→ Dense(32, ReLU) ──┘
wind_dir_sin                  ↓ Tanh
wind_dir_cos            Scaled to ranges:
wind_speed              heading: [-15°, +15°]
turbulence              throttle: [-50, +50] km/h
                        pitch: [-10°, +10°]
```